# IAEA ENDF Download Demo

This notebook demonstrates the ENDF remote download functionality from IAEA Nuclear Data Service.

In [1]:
from kika.endf import (
    fetch_endf,
    list_available_libraries,
    get_cache_info,
    clear_cache,
    IsotopeNotFoundError,
    LibraryNotFoundError,
)

## 1. List Available Libraries

In [2]:
print("Available libraries:")
for lib in list_available_libraries():
    print(f"  - {lib}")

Available libraries:
  - endfb8.1
  - endfb8.0
  - endfb7.1
  - endfb7.0
  - jeff4.0
  - jeff3.3
  - jeff3.2
  - jeff3.1.1
  - jendl5
  - jendl4.0
  - tendl2023
  - tendl2021
  - cendl3.2


## 2. Download Fe-56 from ENDF/B-VIII.1

In [3]:
# Download Fe-56 (first download will take a moment)
print("Downloading Fe-56 from ENDF/B-VIII.1...")
endf_fe56 = fetch_endf("Fe56")
print(f"Downloaded successfully!")
print(f"Type: {type(endf_fe56)}")

C:\Users\Usuario\BaradDur\Dev\kika\kika\endf\parsers\parse_endf.py:90: UserWarning: Skipping MF sections without parsers: [2, 3, 6, 12, 14, 33]. Only parsing: [1, 4]
  warnings.warn(f"Skipping MF sections without parsers: {skipped_mfs}. Only parsing: {parseable_mfs}")


Downloaded successfully!
Type: <class 'kika.endf.classes.endf.ENDF'>


In [4]:
# Second call should be instant (from cache)
import time

start = time.time()
endf_fe56_cached = fetch_endf("Fe56")
elapsed = time.time() - start
print(f"Second call (cached): {elapsed:.4f}s")

Second call (cached): 0.5227s


## 3. Different Isotope Formats

In [5]:
# Using ZAID format
print("Using ZAID format (26056):")
endf_zaid = fetch_endf(26056)
print("Success!")

Using ZAID format (26056):
Success!


In [6]:
# Using hyphenated format
print("Using hyphenated format (Fe-56):")
endf_hyphen = fetch_endf("Fe-56")
print("Success!")

Using hyphenated format (Fe-56):
Success!


## 4. Different Library Formats

In [7]:
# Test different library name formats (all equivalent)
library_formats = [
    "endfb8.1",      # canonical
    "endfb81",       # compact
    "ENDF/B-VIII.1", # official
]

for lib_format in library_formats:
    print(f"Testing library format: '{lib_format}'")
    endf = fetch_endf("Fe56", library=lib_format)
    print(f"  -> Success!")

Testing library format: 'endfb8.1'
  -> Success!
Testing library format: 'endfb81'
  -> Success!
Testing library format: 'ENDF/B-VIII.1'
  -> Success!


## 5. Download from Different Libraries

In [8]:
# Download U-235 from JEFF-4.0
print("Downloading U-235 from JEFF-4.0...")
try:
    endf_u235_jeff = fetch_endf("U235", library="jeff4.0")
    print("Downloaded U-235 from JEFF-4.0!")
except IsotopeNotFoundError as e:
    print(f"Isotope not found: {e}")
except Exception as e:
    print(f"Error: {e}")

C:\Users\Usuario\BaradDur\Dev\kika\kika\endf\parsers\parse_endf.py:90: UserWarning: Skipping MF sections without parsers: [2, 3, 5, 6, 8, 10, 12, 14, 15, 31, 33, 35, 40]. Only parsing: [1, 4]
  warnings.warn(f"Skipping MF sections without parsers: {skipped_mfs}. Only parsing: {parseable_mfs}")


Downloaded U-235 from JEFF-4.0!


## 6. Error Handling

In [9]:
# Test unknown library
try:
    fetch_endf("Fe56", library="unknown_library")
except LibraryNotFoundError as e:
    print(f"LibraryNotFoundError: {e}")

LibraryNotFoundError: Unknown library 'unknown_library'. Available libraries: endfb8.1, endfb8.0, endfb7.1, endfb7.0, jeff4.0, jeff3.3, jeff3.2, jeff3.1.1, jendl5, jendl4.0, tendl2023, tendl2021, cendl3.2


In [10]:
# Test unknown isotope
try:
    fetch_endf("Xx999")
except IsotopeNotFoundError as e:
    print(f"IsotopeNotFoundError: {e}")

ValueError: Unknown element symbol: Xx

## 7. Cache Management

In [11]:
# View cache info
cache_info = get_cache_info()
print(f"Cache directory: {cache_info['cache_dir']}")
print(f"Total files: {cache_info['total_files']}")
print(f"Total size: {cache_info['total_size_mb']} MB")
print("\nFiles by library:")
for lib, info in cache_info['libraries'].items():
    print(f"  {lib}: {info['count']} files, {info['size_bytes'] / 1024:.1f} KB")

Cache directory: C:\Users\Usuario\.kika\endf_cache
Total files: 2
Total size: 33.24 MB

Files by library:
  endfb8.1: 1 files, 26930.0 KB
  jeff4.0: 1 files, 7110.0 KB


In [12]:
# Force re-download (ignore cache)
print("Force re-downloading Fe-56...")
endf_fresh = fetch_endf("Fe56", force_download=True)
print("Done!")

Force re-downloading Fe-56...
Done!


In [ ]:
# Clear specific cache entries (uncomment to run)
# removed = clear_cache(library="endfb8.1")
# print(f"Removed {removed} files from cache")

## 8. Working with Downloaded Data

In [13]:
# The returned object is a standard ENDF object
# You can use all the usual kika functionality
endf = fetch_endf("Fe56")

# Access parsed data (example - depends on your ENDF class structure)
print(f"ENDF object: {endf}")
print(f"Available files: {list(endf.files.keys()) if hasattr(endf, 'files') else 'N/A'}")

ENDF object: ENDF(2 files)
Available files: [1, 4]
